In [10]:
"""
Bayesian Self-Regulating Network (SRN) — PyTorch implementation
================================================================
Architecture
------------
A self-regulating network uses per-layer precision parameters
  alpha  (weight precision) and  beta  (noise/likelihood precision)
drawn from Bayesian linear-regression theory (MacKay, 1992).

Instead of fixed regularisation, the network *learns* alpha and beta
alongside its weights via Type-II maximum likelihood (empirical Bayes).

The ELBO we minimise:

    L = (beta/2) * sum((y - y_hat)^2)    <-- scaled MSE
      + (alpha/2) * sum(w^2)             <-- L2 weight prior
      - (N/2) * log(beta)               <-- normalises likelihood
      - (D/2) * log(alpha)              <-- normalises prior

Self-regulation:
  * if weights grow  -> alpha rises  -> stronger regularisation
  * if residuals shrink -> beta rises -> model gets more confident
  * both converge to stable values when evidence landscape flattens

Dataset
-------
Synthetic regression: y = 2x1 - 3x2 + 0.5x3 + N(0, 0.3^2)
True noise variance = 0.09  =>  theoretical beta* = 1/0.09 ~= 11.1
"""

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    USE_TORCH = True
    print("Using real PyTorch", torch.__version__)
except ImportError:
    USE_TORCH = False
    print("torch not installed -- running numpy finite-diff shim")

import numpy as np
import math

# ── lightweight numpy shim so the script runs without torch ──────────────────
if not USE_TORCH:
    class _P:
        "Minimal Parameter: holds data + gradient."
        def __init__(self, data):
            self.data = np.asarray(data, dtype=np.float64)
            self.grad = None

    class _Linear:
        def __init__(self, in_f, out_f):
            k = math.sqrt(1 / in_f)
            self.weight = _P(np.random.uniform(-k, k, (out_f, in_f)))
            self.bias   = _P(np.zeros(out_f))
        def __call__(self, x):
            return x @ self.weight.data.T + self.bias.data
        def parameters(self): return [self.weight, self.bias]

    class _Adam:
        def __init__(self, params, lr=1e-3):
            self.p, self.lr = params, lr
            self.m  = [np.zeros_like(p.data) for p in params]
            self.v  = [np.zeros_like(p.data) for p in params]
            self.t  = 0
        def zero_grad(self):
            for p in self.p: p.grad = None
        def step(self):
            self.t += 1; b1, b2, eps = 0.9, 0.999, 1e-8
            for i, p in enumerate(self.p):
                g = p.grad if p.grad is not None else np.zeros_like(p.data)
                self.m[i] = b1*self.m[i] + (1-b1)*g
                self.v[i] = b2*self.v[i] + (1-b2)*g**2
                mh = self.m[i]/(1-b1**self.t)
                vh = self.v[i]/(1-b2**self.t)
                p.data -= self.lr * mh / (np.sqrt(vh)+eps)


# ══════════════════════════════════════════════════════════════════════════════
#  Dataset
# ══════════════════════════════════════════════════════════════════════════════
np.random.seed(42)
if USE_TORCH: import torch; torch.manual_seed(42)

N, D       = 400, 8
TRUE_W     = np.array([2.0, -3.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0])
NOISE_STD  = 0.3          # true sigma_noise  =>  true beta* = 1/0.09 ~ 11.1
TRUE_BETA  = 1.0 / NOISE_STD**2

X_np = np.random.randn(N, D)
y_np = X_np @ TRUE_W + np.random.randn(N) * NOISE_STD

split        = int(0.8 * N)
X_tr, X_val  = X_np[:split], X_np[split:]
y_tr, y_val  = y_np[:split], y_np[split:]


# ══════════════════════════════════════════════════════════════════════════════
#  Model
# ══════════════════════════════════════════════════════════════════════════════
HIDDEN = 32

if USE_TORCH:
    import torch, torch.nn as nn, torch.nn.functional as F

    class BayesianSRN(nn.Module):
        def __init__(self, in_f, hidden=HIDDEN):
            super().__init__()
            self.fc1 = nn.Linear(in_f, hidden)
            self.fc2 = nn.Linear(hidden, 1)
            # log-space so softplus keeps them positive
            self.log_alpha = nn.Parameter(torch.tensor(0.54))  # softplus(0.54)~1
            self.log_beta  = nn.Parameter(torch.tensor(0.54))

        def forward(self, x):
            return self.fc2(torch.relu(self.fc1(x))).squeeze(-1)

        def alpha(self): return F.softplus(self.log_alpha)
        def beta(self):  return F.softplus(self.log_beta)

        def weight_sq_sum(self):
            return sum((p**2).sum() for p in [self.fc1.weight, self.fc1.bias,
                                               self.fc2.weight, self.fc2.bias])
        def elbo(self, x, y):
            y_hat = self.forward(x)
            N_    = y.numel()
            D_    = sum(p.numel() for p in [self.fc1.weight, self.fc1.bias,
                                             self.fc2.weight, self.fc2.bias])
            a, b  = self.alpha(), self.beta()
            nll   = (b/2)*((y - y_hat)**2).sum() - (N_/2)*torch.log(b)
            prior = (a/2)*self.weight_sq_sum()    - (D_/2)*torch.log(a)
            return nll + prior

    model = BayesianSRN(D)
    X_tr_t  = torch.tensor(X_tr,  dtype=torch.float32)
    y_tr_t  = torch.tensor(y_tr,  dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)
    optimizer = torch.optim.Adam(
        [{'params': list(model.fc1.parameters()) + list(model.fc2.parameters()), 'lr': 1e-3},
         {'params': [model.log_alpha, model.log_beta], 'lr': 5e-3}]
    )

else:
    # numpy model
    fc1 = _Linear(D, HIDDEN)
    fc2 = _Linear(HIDDEN, 1)
    log_alpha = _P(np.array(0.54))
    log_beta  = _P(np.array(0.54))

    all_params = fc1.parameters() + fc2.parameters() + [log_alpha, log_beta]
    optimizer  = _Adam(all_params, lr=1e-3)

    def softplus(x): return np.log1p(np.exp(x))
    def get_alpha(): return softplus(log_alpha.data.item())
    def get_beta():  return softplus(log_beta.data.item())

    def forward(X):
        h = np.maximum(0, fc1(X))
        return fc2(h).ravel()

    def all_weight_params():
        return [fc1.weight, fc1.bias, fc2.weight, fc2.bias]

    def compute_elbo(X, y):
        y_hat = forward(X)
        N_    = len(y)
        D_    = sum(p.data.size for p in all_weight_params())
        a, b  = get_alpha(), get_beta()
        residuals = np.sum((y - y_hat)**2)
        w_sq      = sum(np.sum(p.data**2) for p in all_weight_params())
        nll   = (b/2)*residuals - (N_/2)*math.log(max(b, 1e-9))
        prior = (a/2)*w_sq      - (D_/2)*math.log(max(a, 1e-9))
        return nll + prior, y_hat

    def finite_diff_grads(X, y, eps=1e-5):
        base, _ = compute_elbo(X, y)
        for p in all_params:
            flat = p.data.ravel()
            grad = np.zeros_like(flat)
            for i in range(len(flat)):
                flat[i] += eps
                plus, _ = compute_elbo(X, y)
                flat[i] -= eps
                grad[i] = (plus - base) / eps
            p.grad = grad.reshape(p.data.shape)


# ══════════════════════════════════════════════════════════════════════════════
#  Training
# ══════════════════════════════════════════════════════════════════════════════
EPOCHS       = 600
LOG_INTERVAL = 100
history      = {k: [] for k in ["epoch","loss","val_mse","alpha","beta"]}

print(f"\n{'─'*65}")
print(f"{'Epoch':>6}  {'ELBO':>10}  {'Val MSE':>9}  {'alpha':>8}  {'beta':>8}")
print(f"{'─'*65}")

for epoch in range(1, EPOCHS + 1):
    if USE_TORCH:
        optimizer.zero_grad()
        loss = model.elbo(X_tr_t, y_tr_t)
        loss.backward()
        optimizer.step()
        loss_v  = loss.item()
        a_v     = model.alpha().item()
        b_v     = model.beta().item()
        with torch.no_grad():
            val_mse = ((model(X_val_t) - y_val_t)**2).mean().item()
    else:
        finite_diff_grads(X_tr, y_tr)
        optimizer.step()
        optimizer.zero_grad()
        loss_v, _ = compute_elbo(X_tr, y_tr)
        a_v       = get_alpha()
        b_v       = get_beta()
        val_mse   = np.mean((forward(X_val) - y_val)**2)

    history["epoch"].append(epoch)
    history["loss"].append(loss_v)
    history["val_mse"].append(val_mse)
    history["alpha"].append(a_v)
    history["beta"].append(b_v)

    if epoch % LOG_INTERVAL == 0 or epoch == 1:
        print(f"{epoch:>6}  {loss_v:>10.4f}  {val_mse:>9.5f}  "
              f"{a_v:>8.4f}  {b_v:>8.4f}")

print(f"{'─'*65}")


# ══════════════════════════════════════════════════════════════════════════════
#  Convergence analysis
# ══════════════════════════════════════════════════════════════════════════════
tail       = max(1, EPOCHS // 5)
a_tail     = np.array(history["alpha"][-tail:])
b_tail     = np.array(history["beta"][-tail:])
a_mean, a_std = float(np.mean(a_tail)), float(np.std(a_tail))
b_mean, b_std = float(np.mean(b_tail)), float(np.std(b_tail))

a_cv = a_std / max(a_mean, 1e-9)   # coefficient of variation
b_cv = b_std / max(b_mean, 1e-9)

THRESH = 0.02  # 2% CV => converged

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  Convergence report  (last {tail} of {EPOCHS} epochs)
╠══════════════════════════════════════════════════════════════════╣
║  alpha  mean = {a_mean:7.4f}   std = {a_std:.2e}   CV = {a_cv:.4f}
║  beta   mean = {b_mean:7.4f}   std = {b_std:.2e}   CV = {b_cv:.4f}
║
║  Theoretical beta* (= 1/sigma^2 = 1/{NOISE_STD}^2) = {TRUE_BETA:.4f}
║  Learned  beta  converged to ~ {b_mean:.4f}
║  Ratio learned/theoretical    = {b_mean/TRUE_BETA:.3f}  (1.0 = perfect)
║
║  alpha converged? {"YES  (CV < 2%)" if a_cv < THRESH else "needs more epochs":<30}
║  beta  converged? {"YES  (CV < 2%)" if b_cv < THRESH else "needs more epochs":<30}
╚══════════════════════════════════════════════════════════════════╝""")

# ASCII plot of alpha & beta trajectories
print("\n  alpha (a) and beta (b) trajectories\n  " + "─"*50)
sample = max(1, EPOCHS // 20)
ep_s   = history["epoch"][::sample]
a_s    = history["alpha"][::sample]
b_s    = history["beta"][::sample]
W = 36
vmin   = min(min(a_s), min(b_s))
vmax   = max(max(a_s), max(b_s))
span   = max(vmax - vmin, 1e-9)

def bar(v, ch):
    pos = int((v - vmin) / span * W)
    return " " * pos + ch + f" {v:.3f}"

for ep, a, b in zip(ep_s, a_s, b_s):
    print(f"  ep{ep:>4}  a|{bar(a, 'A')}")
    print(f"          b|{bar(b, 'B')}")

print(f"\n  [{vmin:.3f}" + " "*W + f"{vmax:.3f}]")

# Validation
final_rmse = math.sqrt(history["val_mse"][-1])
print(f"\n  Final val RMSE : {final_rmse:.4f}   (true noise sigma = {NOISE_STD})")
ok = final_rmse < 2.5 * NOISE_STD
print(f"  Within 2.5x noise floor? {'YES' if ok else 'NO'}")

Using real PyTorch 2.11.0+cu130

─────────────────────────────────────────────────────────────────
 Epoch        ELBO    Val MSE     alpha      beta
─────────────────────────────────────────────────────────────────
     1   2138.5378   13.25009    1.0023    0.9960
   100   1145.6094   10.26449    1.3331    0.7379
   200    413.1826    4.71003    1.6809    0.6065
   300     58.6233    0.86106    2.0231    0.5910
   400    -13.7155    0.40599    2.3543    0.6298
   500    -48.9740    0.33120    2.6782    0.6830
   600    -80.3436    0.28874    2.9954    0.7462
─────────────────────────────────────────────────────────────────

╔══════════════════════════════════════════════════════════════════╗
║  Convergence report  (last 120 of 600 epochs)
╠══════════════════════════════════════════════════════════════════╣
║  alpha  mean =  2.8071   std = 1.10e-01   CV = 0.0392
║  beta   mean =  0.7080   std = 2.16e-02   CV = 0.0305
║
║  Theoretical beta* (= 1/sigma^2 = 1/0.3^2) = 11.1111
║  Learned  b